# CMPE 401 Instructor-defined Project 1

## Multi-Version YOLO Comparison

This notebook trains YOLOv8n, YOLOv9t, and YOLOv10n on the same
VisDrone2019-DET trainest dataset, then compares all three models against
the YOLOv11n baseline.

---


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/YOLOv11-Project') # results are saved here
DRIVE_DIR.mkdir(exist_ok=True)

# unzip the directory containing the dataset
ZIP_PATH = '/content/drive/MyDrive/VisDrone2019-DET-train.zip'
if not Path('/content/VisDrone2019-DET-train/images').exists():
    import zipfile
    print('Extracting dataset to local storage...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done.')
else:
    print('Local dataset already present.')

In [ ]:
%pip install ultralytics numpy matplotlib pandas Pillow tqdm -q

# Environment & Paths

**Prerequisite:** `yolov11.ipynb` must be ran first.


In [ ]:
import os
import time
import random
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from ultralytics import YOLO
import ultralytics

SEED = 15
random.seed(SEED)
np.random.seed(SEED)

EPOCHS = 50
IMGSZ  = 640

BASE_DIR    = DRIVE_DIR # save data we need to persist to Google Drive
DATASET_DIR = BASE_DIR / 'dataset'
yaml_path   = DATASET_DIR / 'dataset.yaml'
RUNS_DIR    = BASE_DIR / 'runs'
VAL_IMG     = DATASET_DIR / 'images' / 'val'
COMPARISON_NAME = 'multi-version-comparison'

print('Project root (Drive) :', BASE_DIR)
print('Dataset yaml         :', yaml_path)
print('Val images           :', len(list(VAL_IMG.glob('*.jpg'))), 'files')
print('Runs dir             :', RUNS_DIR)

## Model Registry

In [ ]:
MODELS = {
    'YOLOv8n':  {'weights': 'yolov8n.pt',  'run_name': 'yolov8n_comparison'},
    'YOLOv9t':  {'weights': 'yolov9t.pt',  'run_name': 'yolov9t_comparison'},
    'YOLOv10n': {'weights': 'yolov10n.pt', 'run_name': 'yolov10n_comparison'},
   # YOLOv11n was trained in yolov11n.ipynb, so we only need to load its results
    'YOLOv11n': {'weights': 'yolo11n.pt',  'run_name': 'yolov11n_baseline'},
}

TO_TRAIN = {k: v for k, v in MODELS.items() if k != 'YOLOv11n'}

# define a consistent color palette used across all plots for e/a model
COLORS = {
    'YOLOv8n' :  '#3c6be1',
    'YOLOv9t' :  '#f78802',
    'YOLOv10n':  '#d462a6',
    'YOLOv11n':  '#00b159',
}

print('Models to train  :', list(TO_TRAIN.keys()))
v11_run = RUNS_DIR / MODELS['YOLOv11n']['run_name']
status  = 'found' if (v11_run / 'results.csv').exists() else 'NOT FOUND. Run yolov11n.ipynb first'
print(f'YOLOv11n run dir : {v11_run} was {status}')

# Train Comparison Models

*Note*: Training is automatically skipped if `results.csv` already exists in the run directory, so this cell is safe to re-run.

In [ ]:
gpu  = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
print(f'GPU  : {gpu.name}')
print(f'VRAM : {vram:.1f} GB')
print(f'CUDA : {torch.version.cuda}')

BATCH = 64 if vram < 70 else 128
print(f'Batch: {BATCH}  (auto-selected for {vram:.0f} GB VRAM)')

train_times = {}

for name, cfg in TO_TRAIN.items():
    run_dir = RUNS_DIR / cfg['run_name']
    results_csv = run_dir / 'results.csv'

    if results_csv.exists():
        print(f'[{name}] Already trained; Skipping ({results_csv})')
        train_times[name] = None
        continue

    print(f'\n{"=" * 60}')
    print(f'  Training {name}  ({cfg["weights"]})')
    print(f'{"=" * 60}')

    model = YOLO(cfg['weights'])
    t0 = time.perf_counter()
    model.train(
        data     = str(yaml_path),
        epochs   = EPOCHS,
        imgsz    = IMGSZ,
        batch    = BATCH,
        workers  = 8,       # parallel dataloaders
        cache    = 'ram',   # load all images into RAM once at the start
        amp      = True,    # Automatic Mixed Precision
        cos_lr   = True,    # cosine LR decay
        device   = 0,
        name     = cfg['run_name'],
        project  = str(RUNS_DIR),
        seed     = SEED,
        verbose  = True,
        exist_ok = True,
    )

    elapsed = time.perf_counter() - t0
    train_times[name] = elapsed
    print(f'\n[{name}] Finished in {elapsed / 60:.1f} min')

train_times.setdefault('YOLOv11n', None)
print('\nAll models ready.')

# Collect & Aggregate Results


In [ ]:
def load_results(run_dir: Path):
    csv_path = run_dir / 'results.csv'
    if not csv_path.exists():
        print(f'  WARNING: {csv_path} not found')
        return None
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    return df


def find_col(df, patterns):
    """
    Return the first column whose name contains any pattern (case-insensitive).
    """
    for p in patterns:
        for c in df.columns:
            if p.lower() in c.lower():
                return c
    return None


def best_epoch_metrics(df: pd.DataFrame) -> dict:
    """
    Extract metrics at the epoch with the highest mAP@0.5:0.95.
    """
    map95_col = find_col(df, ['mAP50-95', 'mAP_0.5:0.95'])
    map50_col = find_col(df, ['mAP50(B)', 'mAP50'])
    if map95_col is None:
        map95_col = map50_col   # fallback if no mAP50-95 column

    best_idx  = df[map95_col].idxmax()
    row = df.loc[best_idx]
    epoch_col = df.columns[0]

    def g(patterns):
        col = find_col(df, patterns)
        return float(row[col]) if col else float('nan')

    return {
        'best_epoch': int(row[epoch_col]),
        'mAP50':      g(['mAP50(B)', 'mAP50']),
        'mAP50_95':   g(['mAP50-95(B)', 'mAP50-95', 'mAP_0.5:0.95']),
        'precision':  g(['precision(B)', 'precision']),
        'recall':     g(['recall(B)', 'recall']),
        'val_box':    g(['val/box_loss', 'val_box']),
        'val_cls':    g(['val/cls_loss', 'val_cls']),
        'val_dfl':    g(['val/dfl_loss', 'val_dfl']),
    }

In [ ]:
# collect the results of the 4 models
all_results = {}  # model_name = {'df', 'metrics', 'weights_mb', 'run_dir'}

for name, cfg in MODELS.items():
    run_dir = RUNS_DIR / cfg['run_name']
    df = load_results(run_dir)
    if df is None:
        continue

    metrics = best_epoch_metrics(df)
    weights_path = run_dir / 'weights' / 'best.pt'
    weights_mb = (weights_path.stat().st_size / 1e6 if weights_path.exists() else float('nan'))

    all_results[name] = {
        'df':         df,
        'metrics':    metrics,
        'weights_mb': weights_mb,
        'run_dir':    run_dir,
    }

    print(f'{name:<10}  mAP50={metrics["mAP50"]:.4f}  '
          f'mAP50-95={metrics["mAP50_95"]:.4f}  '
          f'size={weights_mb:.1f} MB  best_epoch={metrics["best_epoch"]}')

## Summary Table

In [ ]:
FIG_DPI = 130
plt.style.use('seaborn-v0_8-whitegrid')

# build a summary dataframe
rows = []
for name in MODELS:
    if name not in all_results:
        continue

    r = all_results[name]
    row = {'Model': name, **r['metrics'], 'weights_mb': r['weights_mb']}
    row['train_min'] = (
        train_times.get(name) / 60
        if train_times.get(name) is not None else float('nan')
    )
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')

# print summary table and save to CSV
hdr = (f'{"Model":<10} {"mAP@0.5":>9} {"mAP@0.5:0.95":>13}'
       f' {"Precision":>10} {"Recall":>8} {"Size MB":>9} {"Best Ep":>8}')
print('=' * len(hdr))
print(hdr)
print('-' * len(hdr))

for mname, row in summary_df.iterrows():
    print(f'{mname:<10} {row["mAP50"]:>9.4f} {row["mAP50_95"]:>13.4f}'
          f' {row["precision"]:>10.4f} {row["recall"]:>8.4f}'
          f' {row["weights_mb"]:>9.1f} {int(row["best_epoch"]):>8}')

print('=' * len(hdr))

out_csv = BASE_DIR / COMPARISON_NAME / 'yolo_comparison_results.csv'
summary_df.to_csv(out_csv)
print(f'\nComparison results saved to {out_csv}')

## Comparison Graphs & Visualizations

In [ ]:
models_present = [m for m in MODELS if m in all_results]
x = np.arange(len(models_present))
colors = [COLORS[m] for m in models_present]

metrics_cfg = [
    ('mAP50',     'mAP@0.50'),
    ('mAP50_95',  'mAP@0.50:0.95'),
    ('precision', 'Precision'),
    ('recall',    'Recall'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5), dpi=FIG_DPI)
fig.suptitle(
    'YOLO mAP, Precision, and Recall Comparison',
    fontsize=13, fontweight='bold', y=1.02
)

for ax, (metric, label) in zip(axes, metrics_cfg):
    values = [all_results[m]['metrics'][metric] for m in models_present]
    bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.8, width=0.6)
    ax.set_title(label, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models_present, rotation=22, ha='right', fontsize=9)
    finite = [v for v in values if not np.isnan(v)]
    if finite:
        ax.set_ylim(0, min(1.0, max(finite) * 1.22))
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))

    for bar, val in zip(bars, values):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.savefig(BASE_DIR / COMPARISON_NAME / 'comparison_bar_metrics.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('Saved: comparison_bar_metrics.png')

In [ ]:
# mAP learning curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=FIG_DPI)
fig.suptitle('mAP Curve Over Training Period (All Models)',
             fontsize=13, fontweight='bold', y=1.02)

for name in models_present:
    df = all_results[name]['df']
    color = COLORS[name]
    ep = df[df.columns[0]].values

    col50 = find_col(df, ['mAP50(B)', 'mAP50'])
    col95 = find_col(df, ['mAP50-95(B)', 'mAP50-95'])

    if col50:
        axes[0].plot(ep, df[col50].values, label=name, color=color, linewidth=2)
    if col95:
        axes[1].plot(ep, df[col95].values, label=name, color=color, linewidth=2)

for ax, title, ylabel in [
    (axes[0], 'mAP@0.5 vs Epoch',      'mAP@0.5'),
    (axes[1], 'mAP@0.5:0.95 vs Epoch', 'mAP@0.5:0.95'),
]:
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(BASE_DIR / COMPARISON_NAME / 'comparison_map_curves.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('Saved: comparison_map_curves.png')

In [ ]:
# validation loss curves
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), dpi=FIG_DPI)
fig.suptitle('Validation Loss Curves (All Models)',
             fontsize=13, fontweight='bold', y=1.02)

loss_cfg = [
    (['val/box_loss', 'val_box'], axes[0], 'Box Loss'),
    (['val/cls_loss', 'val_cls'], axes[1], 'Class Loss'),
    (['val/dfl_loss', 'val_dfl'], axes[2], 'DFL Loss'),
]

for patterns, ax, title in loss_cfg:
    for name in models_present:
        df = all_results[name]['df']
        col = find_col(df, patterns)
        if col:
            ep = df[df.columns[0]].values
            ax.plot(ep, df[col].values, label=name, color=COLORS[name], linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(BASE_DIR / COMPARISON_NAME / 'comparison_val_loss.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('Saved: comparison_val_loss.png')

In [ ]:
# model size vs accuracy & speed breakdown
speed_ms = {}
detailed_speed = {}

# calculate the total weight for each graph
for name in models_present:
    weights = all_results[name]['run_dir'] / 'weights' / 'best.pt'
    if not weights.exists():
        speed_ms[name] = float('nan')
        detailed_speed[name] = {'prep': float('nan'), 'inf': float('nan'), 'post': float('nan'), 'total': float('nan')}
        continue

    m = YOLO(str(weights))
    val_result = m.val(data=str(yaml_path), verbose=False, plots=False, device=0)

    prep  = val_result.speed['preprocess']
    inf   = val_result.speed['inference']
    post  = val_result.speed['postprocess']
    total = prep + inf + post

    detailed_speed[name] = {'prep': prep, 'inf': inf, 'post': post, 'total': total}
    speed_ms[name] = total
    print(f'  {name}: {inf:.2f} ms inference  |  {total:.2f} ms total')

# plot the graphs
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=FIG_DPI)
fig.suptitle('Model Size vs Accuracy and Speed Breakdown', fontsize=13, fontweight='bold', y=1.02)

# model size vs mAP@0.50
ax = axes[0]
for name in models_present:
    r = all_results[name]
    ax.scatter(r['weights_mb'], r['metrics']['mAP50'],
               color=COLORS[name], s=140, zorder=5, label=name,
               edgecolors='white', linewidths=0.8)

    ax.annotate(name,
                xy=(r['weights_mb'], r['metrics']['mAP50']),
                xytext=(5, 4), textcoords='offset points', fontsize=9)

ax.set_xlabel('Model Size (MB)', fontsize=11)
ax.set_ylabel('mAP@0.5', fontsize=11)
ax.set_title('Model Size vs. Accuracy', fontweight='bold')
ax.legend(fontsize=9)

# speed breakdown
ax3 = axes[1]
if detailed_speed:
    names_d = list(detailed_speed.keys())
    prep_vals = [detailed_speed[m]['prep']  for m in names_d]
    inf_vals = [detailed_speed[m]['inf']   for m in names_d]
    post_vals = [detailed_speed[m]['post']  for m in names_d]

    ax3.bar(names_d, prep_vals, label='Preprocess',  color='lightblue')
    ax3.bar(names_d, inf_vals,  label='Inference',   color='steelblue',
            bottom=prep_vals)

    bottom_post = [p + i for p, i in zip(prep_vals, inf_vals)]
    ax3.bar(names_d, post_vals, label='Postprocess', color='darkblue',
            bottom=bottom_post)

    max_tot = 0
    for i, m in enumerate(names_d):
        tot = detailed_speed[m]['total']
        max_tot = max(max_tot, tot)
        ax3.text(i, tot + (tot * 0.05), f'{tot:.1f} ms',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax3.set_ylim(0, max_tot * 1.35)
    ax3.set_xticks(range(len(names_d)))
    ax3.set_xticklabels(names_d, rotation=15, ha='right')
    ax3.set_ylabel('Time (ms)', fontsize=11)
    ax3.set_title('Speed Breakdown (Preprocess + Inference + Postprocess)', fontweight='bold')
    ax3.legend()

plt.tight_layout()
plt.savefig(BASE_DIR / COMPARISON_NAME / 'comparison_size_speed_train.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('Saved: comparison_size_speed_train.png')


## Automated Analysis & Conclusions

In [ ]:
# ── Automated comparison summary ─────────────────────────────────────────────
ranked_map50 = summary_df['mAP50'].sort_values(ascending=False)
ranked_map95 = summary_df['mAP50_95'].sort_values(ascending=False)
ranked_prec = summary_df['precision'].sort_values(ascending=False)
ranked_recall = summary_df['recall'].sort_values(ascending=False)
ranked_size = summary_df['weights_mb'].sort_values()

best_map50 = ranked_map50.index[0]
best_map95 = ranked_map95.index[0]
best_prec = ranked_prec.index[0]
best_recall = ranked_recall.index[0]
smallest = ranked_size.index[0]

valid_speed = {k: v for k, v in speed_ms.items() if not np.isnan(v)}
fastest = min(valid_speed, key=valid_speed.get) if valid_speed else 'N/A'

sep = '=' * 64
print(sep)
print('  MULTI-VERSION YOLO COMPARISON SUMMARY')
print(sep)
print(f'  Best mAP@0.5       : {best_map50} ({ranked_map50.iloc[0]:.4f})')
print(f'  Best mAP@0.5:0.95  : {best_map95} ({ranked_map95.iloc[0]:.4f})')
print(f'  Best Precision     : {best_prec} ({ranked_prec.iloc[0]:.4f})')
print(f'  Best Recall        : {best_recall} ({ranked_recall.iloc[0]:.4f})')
print(f'  Smallest model     : {smallest} ({ranked_size.iloc[0]:.1f} MB)')
spd = valid_speed.get(fastest, float('nan'))
print(f'  Fastest inference  : {fastest} ({spd:.2f} ms/img)')
print('-' * 64)
print(f'  mAP@0.5 rank       : {" > ".join(ranked_map50.index.tolist())}')
print(f'  mAP@0.5:0.95 rank  : {" > ".join(ranked_map95.index.tolist())}')
print('-' * 64)